# RAG System Evaluation
The objective of this notebook is to evaluate the performance of the Retrieval-Augmented Generation (RAG) system using a set of university-related questions.

The evaluation compares the quality of answers generated by the RAG pipeline against expected information contained within the document collection.

The results will be used to assess whether Retrieval-Augmented Generation improves the accuracy, relevance and grounding of responses compared to relying on a language model alone.

In [29]:
# Install required libraries

!pip install -q pypdf
!pip install -q sentence-transformers
!pip install -q faiss-cpu
!pip install -q transformers
!pip install -q torch
!pip install -q langchain-text-splitters

In [30]:
# ==========================================================
# Imports
# ==========================================================
# These libraries are used for:
# - PDF processing
# - Text chunking
# - Embedding generation
# - Vector storage and retrieval
# - Data analysis and evaluation

from pathlib import Path

from pypdf import PdfReader

from langchain_text_splitters import RecursiveCharacterTextSplitter

from sentence_transformers import SentenceTransformer

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM
)

import faiss
import numpy as np
import pandas as pd

In [31]:
from pathlib import Path

pdf_files = list(
    Path("/kaggle/input/datasets/makhosazanemthethwa/cos802-university-documents")
    .glob("*.pdf")
)

print(f"Documents Loaded: {len(pdf_files)}")

for pdf in pdf_files:
    print(pdf.name)

Documents Loaded: 9
COS802_Project_Description.pdf
COS802_Study_Guide_2026.pdf
Code_of_Conduct_for_Students.pdf
EBIT_Faculty_Rules_2026.pdf
General_Academic_Regulations_2026.pdf
Policy_on_Examinations_and_Related_Matters.pdf
Programme_Yearbook_12254017.pdf
UKZN_Student_Rulebook_2026.pdf
UP_Information_Guide_2026.pdf


In [32]:
documents = []

for pdf_file in pdf_files:

    reader = PdfReader(str(pdf_file))

    text = ""

    for page in reader.pages:

        page_text = page.extract_text()

        if page_text:
            text += page_text + "\n"

    documents.append({
        "source": pdf_file.name,
        "text": text
    })

In [33]:
# Create a text splitter for chunking university documents

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

print("Text splitter created")


Text splitter created


In [34]:
chunks = []
sources = []

for doc in documents:

    split_chunks = text_splitter.split_text(
        doc["text"]
    )

    chunks.extend(split_chunks)

    sources.extend(
        [doc["source"]] * len(split_chunks)
    )

print(f"Total Chunks: {len(chunks)}")
print(f"Total Sources: {len(sources)}")

Total Chunks: 1062
Total Sources: 1062


In [35]:
# Display the first chunk and its source

print("SOURCE:")
print(sources[0])

print("\nCHUNK:")
print(chunks[0][:1000])

SOURCE:
COS802_Project_Description.pdf

CHUNK:
School
 
of
 
Information
 
Technology
 
 
Department
 
of
 
Computer
 
Science
 
 
COS802
 
 
Course
 
Final
 
Project
  
 
 
Lecturer
 
Sthembiso
 
Mkhwanazi
 
 
Deadline:
 
9
 
and
 
20
  
November
 
2026


In [36]:
# Count chunks per source document

import pandas as pd

source_counts = pd.Series(sources).value_counts()

print(source_counts)

General_Academic_Regulations_2026.pdf             455
UP_Information_Guide_2026.pdf                     187
UKZN_Student_Rulebook_2026.pdf                    147
Policy_on_Examinations_and_Related_Matters.pdf    137
EBIT_Faculty_Rules_2026.pdf                        57
Programme_Yearbook_12254017.pdf                    27
Code_of_Conduct_for_Students.pdf                   21
COS802_Study_Guide_2026.pdf                        20
COS802_Project_Description.pdf                     11
Name: count, dtype: int64


In [37]:
# Load the embedding model

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully


In [38]:
# Generate embeddings for all chunks

embeddings = embedding_model.encode(
    chunks,
    show_progress_bar=True
)

embeddings = np.array(embeddings).astype("float32")

print(f"Embeddings Generated: {len(embeddings)}")
print(f"Embedding Dimension: {embeddings.shape[1]}")


Batches:   0%|          | 0/34 [00:00<?, ?it/s]

Embeddings Generated: 1062
Embedding Dimension: 384


In [39]:
# Create a FAISS index

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embeddings)

print(f"Vectors Stored: {index.ntotal}")

Vectors Stored: 1062


In [40]:
# Retrieve chunks together with their source documents

def retrieve_context(question, top_k=5):

    query_embedding = embedding_model.encode([question])

    query_embedding = np.array(
        query_embedding
    ).astype("float32")

    distances, indices = index.search(
        query_embedding,
        top_k
    )

    results = []

    for idx in indices[0]:

        results.append(
            {
                "source": sources[idx],
                "chunk": chunks[idx]
            }
        )

    return results

In [41]:
question = "What is the prerequisite for COS802?"

results = retrieve_context(question)

for i, result in enumerate(results, start=1):

    print(f"\n===== Result {i} =====")
    print("Source:", result["source"])
    print()
    print(result["chunk"][:1000])



===== Result 1 =====
Source: COS802_Project_Description.pdf

1.
 
Project
 
Expectations
 
3
 
2.
 
Deliverables
 
3
 
Project
 
Proposal
 
[Max
 
2
 
pages,
 
Arial
 
size
 
11]
 
4
 
Project
 
Report
 
4
 
Format
 
4
 
Expected
 
Sections
 
4
 
Project
 
Presentation
 
4
 
3.
 
Rubrics
 
4
 
4.
 
Submission
 
Instructions
 
and
 
Deadline
 
4
 
5.
 
Plagiarism
 
and
 
Generative
 
AI
 
5
 
LLM/Generative
 
AI
 
Use
 
(We
 
will
 
use
 
amended
 
2025
 
ACL
 
Guidelines)
 
5
 
Guidelines
 
for
 
Generative
 
Assistance
 
in
 
Authorship
 
(amended
 
for
 
COS802)
 
5
 
 
 
1.
 
Project
 
Expectations
 
 
One
 
of
 
the
 
main
 
goals
 
of
 
COS
 
802
 
is
 
to
 
help
 
you
 
get
 
comfortable
 
with
 
Natural
 
Language
 
Processing
 
(NLP)
 
by
 
applying
 
state-of-the-art
 
methods
 
to
 
real-world
 
problems.
 
The
 
final
 
project
 
is
 
the
 
main
 
output
 
of
 
this
 
module
 
and
 
gives
 
you
 
the
 
opportunity
 
to
 
work
 
independently
 
on
 
a
 
focused
 
research
 


In [42]:
for rank, result in enumerate(results, start=1):

    print(f"\nRank {rank}")

    print("Source:", result["source"])


Rank 1
Source: COS802_Project_Description.pdf

Rank 2
Source: COS802_Study_Guide_2026.pdf

Rank 3
Source: COS802_Study_Guide_2026.pdf

Rank 4
Source: COS802_Study_Guide_2026.pdf

Rank 5
Source: COS802_Study_Guide_2026.pdf


## Observation

Although the correct source document was successfully retrieved, it was not always ranked first. In several cases the correct answer appeared within the top-k retrieval results rather than the highest-ranked result.

In [43]:
questions = [
    "What is a supplementary examination?",
    "What constitutes student misconduct?",
    "What support services are available to students?"
]

In [44]:
results = retrieve_context(question)

for rank, result in enumerate(results, start=1):
    print(rank, result["source"])

1 COS802_Project_Description.pdf
2 COS802_Study_Guide_2026.pdf
3 COS802_Study_Guide_2026.pdf
4 COS802_Study_Guide_2026.pdf
5 COS802_Study_Guide_2026.pdf


In [45]:
question = "What is the prerequisite for COS802?"

results = retrieve_context(question)

for rank, result in enumerate(results, start=1):

    print(f"\n===== Rank {rank} =====")
    print("Source:", result["source"])
    print()
    print(result["chunk"][:1500])


===== Rank 1 =====
Source: COS802_Project_Description.pdf

1.
 
Project
 
Expectations
 
3
 
2.
 
Deliverables
 
3
 
Project
 
Proposal
 
[Max
 
2
 
pages,
 
Arial
 
size
 
11]
 
4
 
Project
 
Report
 
4
 
Format
 
4
 
Expected
 
Sections
 
4
 
Project
 
Presentation
 
4
 
3.
 
Rubrics
 
4
 
4.
 
Submission
 
Instructions
 
and
 
Deadline
 
4
 
5.
 
Plagiarism
 
and
 
Generative
 
AI
 
5
 
LLM/Generative
 
AI
 
Use
 
(We
 
will
 
use
 
amended
 
2025
 
ACL
 
Guidelines)
 
5
 
Guidelines
 
for
 
Generative
 
Assistance
 
in
 
Authorship
 
(amended
 
for
 
COS802)
 
5
 
 
 
1.
 
Project
 
Expectations
 
 
One
 
of
 
the
 
main
 
goals
 
of
 
COS
 
802
 
is
 
to
 
help
 
you
 
get
 
comfortable
 
with
 
Natural
 
Language
 
Processing
 
(NLP)
 
by
 
applying
 
state-of-the-art
 
methods
 
to
 
real-world
 
problems.
 
The
 
final
 
project
 
is
 
the
 
main
 
output
 
of
 
this
 
module
 
and
 
gives
 
you
 
the
 
opportunity
 
to
 
work
 
independently
 
on
 
a
 
focused
 
research
 
pr

## Evaluation Table to use in presentation

In [49]:
# Create a dataframe to store evaluation results

import pandas as pd

evaluation_results = pd.DataFrame(
    columns=[
        "Question",
        "Expected Source",
        "Top Source Retrieved",
        "Correct Source in Top 5",
        "Correct Answer Retrieved",
        "Top 5 Sources",
        "RAG Answer",
        "Ranking Notes",
        "Comments"
    ]
)

evaluation_results

,Question,Expected Source,Top Source Retrieved,Correct Source in Top 5,Correct Answer Retrieved,Top 5 Sources,RAG Answer,Ranking Notes,Comments


In [50]:
evaluation_questions = [

    {
        "question": "What is the prerequisite for COS802?",
        "expected_source": "COS802_Study_Guide_2026.pdf"
    },

    {
        "question": "What is the minimum entry requirement for admission to a master's degree?",
        "expected_source": "General_Academic_Regulations_2026.pdf"
    },

    {
        "question": "What is a supplementary examination?",
        "expected_source": "Policy_on_Examinations_and_Related_Matters.pdf"
    },

    {
        "question": "What constitutes student misconduct?",
        "expected_source": "Code_of_Conduct_for_Students.pdf"
    },

    {
        "question": "What support services are available to students?",
        "expected_source": "UP_Information_Guide_2026.pdf"
    },

    {
        "question": "What is an honours degree?",
        "expected_source": "General_Academic_Regulations_2026.pdf"
    },

    {
        "question": "What qualification is required for admission to an honours degree?",
        "expected_source": "General_Academic_Regulations_2026.pdf"
    },

    {
        "question": "Can admission to an honours programme be refused?",
        "expected_source": "General_Academic_Regulations_2026.pdf"
    },

    {
        "question": "What is a professional master's degree?",
        "expected_source": "General_Academic_Regulations_2026.pdf"
    },

    {
        "question": "What are the deliverables for the COS802 project?",
        "expected_source": "COS802_Project_Description.pdf"
    }

]

In [52]:
# Retrieve chunks together with source information

def retrieve_context(question, top_k=5):

    query_embedding = embedding_model.encode([question])

    query_embedding = np.array(
        query_embedding
    ).astype("float32")

    distances, indices = index.search(
        query_embedding,
        top_k
    )

    results = []

    for idx in indices[0]:

        results.append(
            {
                "source": sources[idx],
                "chunk": chunks[idx]
            }
        )

    return results

In [54]:
# Evaluate retrieval performance only

for item in evaluation_questions:

    question = item["question"]

    expected_source = item["expected_source"]

    results = retrieve_context(question)

    top_source = results[0]["source"]

    top_5_sources = [
        r["source"]
        for r in results
    ]

    correct_source = expected_source in top_5_sources

    evaluation_results.loc[len(evaluation_results)] = [
        question,
        expected_source,
        top_source,
        "Yes" if correct_source else "No",
        "",
        ", ".join(top_5_sources),
        "",
        "",
        ""
    ]

In [55]:
evaluation_results[
    [
        "Question",
        "Expected Source",
        "Top Source Retrieved",
        "Correct Source in Top 5"
    ]
]

,Question,Expected Source,Top Source Retrieved,Correct Source in Top 5
0,What is the prerequisite for COS802?,COS802_Study_Guide_2026.pdf,COS802_Project_Description.pdf,Yes
1,What is the minimum entry requirement for admi...,General_Academic_Regulations_2026.pdf,General_Academic_Regulations_2026.pdf,Yes
2,What is a supplementary examination?,Policy_on_Examinations_and_Related_Matters.pdf,General_Academic_Regulations_2026.pdf,Yes
3,What constitutes student misconduct?,Code_of_Conduct_for_Students.pdf,General_Academic_Regulations_2026.pdf,No
4,What support services are available to students?,UP_Information_Guide_2026.pdf,General_Academic_Regulations_2026.pdf,Yes
5,What is an honours degree?,General_Academic_Regulations_2026.pdf,General_Academic_Regulations_2026.pdf,Yes
6,What qualification is required for admission t...,General_Academic_Regulations_2026.pdf,General_Academic_Regulations_2026.pdf,Yes
7,Can admission to an honours programme be refused?,General_Academic_Regulations_2026.pdf,General_Academic_Regulations_2026.pdf,Yes
8,What is a professional master's degree?,General_Academic_Regulations_2026.pdf,General_Academic_Regulations_2026.pdf,Yes
9,What are the deliverables for the COS802 project?,COS802_Project_Description.pdf,COS802_Project_Description.pdf,Yes


In [56]:
# Measure source retrieval accuracy

top5_correct = (
    evaluation_results[
        "Correct Source in Top 5"
    ] == "Yes"
).sum()

total_questions = len(
    evaluation_results
)

accuracy = (
    top5_correct /
    total_questions
) * 100

print(
    f"Top-5 Retrieval Accuracy: {accuracy:.2f}%"
)

Top-5 Retrieval Accuracy: 90.00%


In [ ]:
evaluation_results.to_csv(
    "retrieval_evaluation_results.csv",
    index=False
)

print(
    "Retrieval evaluation results saved."
)

In [60]:
evaluation_results = (
    evaluation_results
    .drop_duplicates(subset=["Question"])
    .reset_index(drop=True)
)

evaluation_results

,Question,Expected Source,Top Source Retrieved,Correct Source in Top 5,Correct Answer Retrieved,Top 5 Sources,RAG Answer,Ranking Notes,Comments
0,What is the prerequisite for COS802?,COS802_Study_Guide_2026.pdf,COS802_Project_Description.pdf,Yes,,"COS802_Project_Description.pdf, COS802_Study_G...",,,
1,What is the minimum entry requirement for admi...,General_Academic_Regulations_2026.pdf,General_Academic_Regulations_2026.pdf,Yes,,"General_Academic_Regulations_2026.pdf, General...",,,
2,What is a supplementary examination?,Policy_on_Examinations_and_Related_Matters.pdf,General_Academic_Regulations_2026.pdf,Yes,,"General_Academic_Regulations_2026.pdf, General...",,,
3,What constitutes student misconduct?,Code_of_Conduct_for_Students.pdf,General_Academic_Regulations_2026.pdf,No,,"General_Academic_Regulations_2026.pdf, UKZN_St...",,,
4,What support services are available to students?,UP_Information_Guide_2026.pdf,General_Academic_Regulations_2026.pdf,Yes,,"General_Academic_Regulations_2026.pdf, UP_Info...",,,
5,What is an honours degree?,General_Academic_Regulations_2026.pdf,General_Academic_Regulations_2026.pdf,Yes,,"General_Academic_Regulations_2026.pdf, General...",,,
6,What qualification is required for admission t...,General_Academic_Regulations_2026.pdf,General_Academic_Regulations_2026.pdf,Yes,,"General_Academic_Regulations_2026.pdf, General...",,,
7,Can admission to an honours programme be refused?,General_Academic_Regulations_2026.pdf,General_Academic_Regulations_2026.pdf,Yes,,"General_Academic_Regulations_2026.pdf, General...",,,
8,What is a professional master's degree?,General_Academic_Regulations_2026.pdf,General_Academic_Regulations_2026.pdf,Yes,,"General_Academic_Regulations_2026.pdf, General...",,,
9,What are the deliverables for the COS802 project?,COS802_Project_Description.pdf,COS802_Project_Description.pdf,Yes,,"COS802_Project_Description.pdf, COS802_Study_G...",,,


In [61]:
presentation_results = evaluation_results[
    [
        "Question",
        "Top Source Retrieved",
        "Correct Source in Top 5"
    ]
]

presentation_results

,Question,Top Source Retrieved,Correct Source in Top 5
0,What is the prerequisite for COS802?,COS802_Project_Description.pdf,Yes
1,What is the minimum entry requirement for admi...,General_Academic_Regulations_2026.pdf,Yes
2,What is a supplementary examination?,General_Academic_Regulations_2026.pdf,Yes
3,What constitutes student misconduct?,General_Academic_Regulations_2026.pdf,No
4,What support services are available to students?,General_Academic_Regulations_2026.pdf,Yes
5,What is an honours degree?,General_Academic_Regulations_2026.pdf,Yes
6,What qualification is required for admission t...,General_Academic_Regulations_2026.pdf,Yes
7,Can admission to an honours programme be refused?,General_Academic_Regulations_2026.pdf,Yes
8,What is a professional master's degree?,General_Academic_Regulations_2026.pdf,Yes
9,What are the deliverables for the COS802 project?,COS802_Project_Description.pdf,Yes


In [62]:
summary_results = pd.DataFrame(
    {
        "Metric": [
            "Documents Indexed",
            "Chunks Generated",
            "Embedding Dimension",
            "Top-5 Retrieval Accuracy"
        ],
        "Value": [
            9,
            1062,
            384,
            "90%"
        ]
    }
)

summary_results

,Metric,Value
0,Documents Indexed,9
1,Chunks Generated,1062
2,Embedding Dimension,384
3,Top-5 Retrieval Accuracy,90%


## Observations
Ten evaluation questions were tested using the multi-document retrieval system.

The correct source document was retrieved within the top five results for 90% of the evaluation questions. Retrieval performance was strongest for questions relating to admissions, honours degrees, master's programmes and COS802 project requirements.

The retrieval system occasionally ranked a related document above the most relevant source document. For example, the COS802 prerequisite question retrieved the project description as the highest-ranked document while the correct prerequisite information appeared within the COS802 Study Guide.

These results indicate that semantic retrieval successfully identified relevant source material across the document collection, although ranking quality can still be improved.

## Conclusion
The evaluation demonstrated that the multi-document retrieval system can successfully identify relevant information from a university document collection.

Using embeddings generated by the all-MiniLM-L6-v2 model and a FAISS vector store, the system achieved high retrieval performance across a range of academic, administrative and module-specific questions.

Although retrieval ranking was not always optimal, the correct source document was located within the top retrieval results for most evaluation questions. These findings suggest that the retrieval component provides a strong foundation for Retrieval-Augmented Generation and supports the use of institutional knowledge sources for question answering.